<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: /content/FlyRank_ML_Task1/FlyRank_ML_Task1


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

"""
Lane chosen: Lane 2 — Refresh / Content Opportunity Scoring

I'm picking this lane because I already got a first taste of it in Assignment 1
(Notebook 2): I built a simple hand-written refresh rule, then fit a depth-2
decision tree, and saw the tree beat my rule on Precision@50 (0.540 in-sample).
That result made me want to go deeper on the same underlying question — which
pages deserve review first — but do it properly this time: with a clearer
future-looking label, real baseline reason codes, and honest validation
instead of an in-sample toy comparison.

This is provisional. I may adjust scope (e.g. narrow to one specific action
type, or lean more on CTR-adjusted signals from Lane 4) once I've spent more
time in the warehouse data, but the core direction — rank content for review,
don't just describe it — is what I want to build toward for the capstone.
"""

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

"""
Research question: Which content pages should a review team look at first,
given limited time, based on evidence of decline, staleness, or missed
opportunity?

Unit of analysis: One page (one content_id / content_hash_id), evaluated at
a point in time using its trailing 90-day performance window.

The decision this improves: Right now, a reviewer has to manually scan a
large inventory of pages to decide which ones are worth their limited time.
This project turns that into a ranked list, ordered by evidence-backed
priority, so the highest-value pages surface first.

The action someone takes: A content reviewer opens the top of the ranked
queue and, for each flagged page, does one of: refresh the content, fix low
CTR (title/meta), investigate a decline, or leave it alone (monitor). The
reason codes attached to each page tell them why it was flagged.

Cost of a wrong recommendation:
- False positive (page flagged but didn't need it): wastes a reviewer's
  limited time — the real cost is opportunity cost, since another page that
  genuinely needed attention gets pushed down the queue instead.
- False negative (a genuinely declining page never gets flagged): the page
  keeps losing visibility/traffic silently, which is the more expensive
  failure mode since it compounds the longer it goes unnoticed.

Because false negatives are costlier here, I'll care about recall on
high-value pages, not just precision.

Why data/ML helps at all: A reviewer eyeballing a spreadsheet can't reliably
compare thousands of pages across five or six correlated signals
(impressions, position, CTR, freshness, engagement) at once, consistently,
without bias toward whichever metric they looked at last. A transparent
baseline score plus a validated model does that comparison consistently and
at scale — but only if it can be proven to beat a simple rule on real
held-out data, not just described.
"""
"""
One-paragraph frame:

For a content reviewer deciding which pages to review first, we will build a
ranked priority list from FlyRank's search/content performance data,
scoring pages by likelihood of needing review, measured by Precision@K
(e.g. Precision@50, matching real reviewer capacity). A wrong call costs
either wasted reviewer time (false positive) or a silently worsening page
that goes unflagged (false negative) — the latter is costlier since it
compounds. A plain rule isn't enough because relevant signals (impressions,
position, CTR, freshness, engagement) interact in ways that are hard to
weigh consistently by eye across thousands of pages. We will claim only
observed / directional / decision-support results.
"""

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: pages that are both visible AND stale — the core Lane 2 opportunity
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"Stale + visible pages: {len(stale_visible)} of {len(df)} "
      f"({len(stale_visible)/len(df):.1%})")

# Number 2: declining pages that also have real demand (not noise)
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"Declining pages with real demand: {len(declining_with_demand)} of {len(df)} "
      f"({len(declining_with_demand)/len(df):.1%})")

# Number 3: page-one decay risk — old content sitting in a strong position
decay_risk = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10) &
                 (df["content_age_days"] >= 180)]
print(f"Page-one decay risk pages: {len(decay_risk)} of {len(df)} "
      f"({len(decay_risk)/len(df):.1%})")

print()
print("Takeaway: these three reason codes behave very differently in this")
print("starter sample. 'Declining with demand' (43.8%) and 'page-one decay")
print("risk' (23.6%) both fire on a meaningful, usable slice of the data —")
print("large enough to matter, small enough to still be a priority signal")
print("rather than 'everything.' 'Stale + visible' is surprisingly rare here")
print("(17 pages, 0.1%) — likely because impressions_90d >= 500 is a strict")
print("bar on a small anonymized sample. This tells me the stale+visible")
print("threshold needs tuning (e.g. a lower impression bar) before it's")
print("useful, and that threshold choices genuinely change what 'worth")
print("reviewing' means — which is itself a finding, not just a caveat.")
print("This is directional evidence from the starter slice only; the full")
print("warehouse data will likely shift these proportions and should be")
print("re-checked before locking in thresholds.")

Stale + visible pages: 17 of 30000 (0.1%)
Declining pages with real demand: 13152 of 30000 (43.8%)
Page-one decay risk pages: 7076 of 30000 (23.6%)

Takeaway: these three reason codes behave very differently in this
starter sample. 'Declining with demand' (43.8%) and 'page-one decay
risk' (23.6%) both fire on a meaningful, usable slice of the data —
large enough to matter, small enough to still be a priority signal
rather than 'everything.' 'Stale + visible' is surprisingly rare here
(17 pages, 0.1%) — likely because impressions_90d >= 500 is a strict
bar on a small anonymized sample. This tells me the stale+visible
threshold needs tuning (e.g. a lower impression bar) before it's
useful, and that threshold choices genuinely change what 'worth
reviewing' means — which is itself a finding, not just a caveat.
This is directional evidence from the starter slice only; the full
warehouse data will likely shift these proportions and should be
re-checked before locking in thresholds.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

"""
What I can claim right now:
- On this starter sample, a meaningful share of pages show measurable
  staleness, decline-with-demand, or decay-risk patterns — this is an
  observed fact about the sample, not a prediction.
- These patterns are directional signals worth prioritizing for human
  review, not proof that fixing them will improve performance.

What I cannot claim:
- I cannot claim that refreshing a flagged page will cause a recovery —
  that requires a controlled experiment, which this data doesn't provide.
- I cannot claim these percentages hold on the full warehouse — the starter
  dataset is a small, anonymized slice and may not be representative.
- I cannot use FlyRank's own product scores (health_score, priority_score,
  etc.) as ground truth, since they aren't in this dataset and should never
  be used as a feature or label if encountered later — that would just
  teach a model to copy an existing rule rather than discover anything new.
- I will not attribute any result to "how Google's algorithm works" — only
  to observed patterns in FlyRank's own tracked data.
"""
"""
Note: the starter dataset's trend_direction label is a current-window bucket,
not a future observed outcome — this skill file (framing-ml-problems) flags
that as a weak target. A stronger capstone target will need to be a genuinely
future-observed outcome (e.g. decline over the next 30 days), not a rule-
derived label.
"""
"""
Data-quality note (from flyrank-data skill): avg_position = 0 in this
dataset means 'no position data', not an actual top rank — my decay-risk
filter already excludes these (avg_position > 0), so that count isn't
inflated by missing-data rows. Also confirming: trend_direction was used
here only as a descriptive EDA filter to count pages, never as a model
feature — the skill file flags trend_direction/trend_pct as label-derived
fields that must never be used as inputs to any future model in this lane.
"""

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.